# 16장 RNN과 어텐션을 사용한 자연어 처리

# 1. Char-RNN으로 셰익스피어 문체 생성

## 목표
- `Char-RNN`을 이용해 셰익스피어 문체 테스트 생성

## 훈련 데이터셋 생성
- 셰익스피어 작품을 훈련 데이터로 사용

In [1]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

import sys
# 코랩의 경우 나눔 폰트를 설치합니다.
if 'google.colab' in sys.modules:
    !sudo apt-get -qq -y install fonts-nanum
    import matplotlib.font_manager as fm
    font_files = fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum'])
    for fpath in font_files:
        fm.fontManager.addfont(fpath)

# 나눔 폰트를 사용합니다.
import matplotlib

matplotlib.rc('font', family='NanumBarunGothic')
matplotlib.rcParams['axes.unicode_minus'] = False

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package fonts-nanum.
(Reading database ... 126380 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...


### 데이터셋 로드

- `Char-RNN` 프로젝트에서 셰익스피어 작품 다운로드

In [2]:
import tensorflow as tf

shakespeare_url = "https://homl.info/shakespeare"
filepath = tf.keras.utils.get_file("shakespeare.txt", shakespeare_url)
with open(filepath) as f:
    shakespeare_text = f.read()

### 텍스트 샘플 표시
print(shakespeare_text[:80])

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


- 39개의 고유 문자 소문자 변환 후 모두 표시

In [3]:
"".join(sorted(set(shakespeare_text.lower())))

"\n !$&',-.3:;?abcdefghijklmnopqrstuvwxyz"

### 텍스트 인코딩
> 신경망 모델이 텍스트 데이터를 이해하고 처리할 수 잇도록 텍스트를 숫자 형태(정수 ID)로 변환하는 필수적인 전처리 과정

- 각 문자는 2부터 시작하는 정수 매핑
  - `패딩 토큰`을 위해 0 사용
  - `알려지지 않은 문자`를 위해 1 사용
  
- `tf.keras.layers.TextVectorization` 층
  - `split="character"`로 문자 수준 인코딩
  - `standardize="lower"`로 텍스트 소문자화

In [4]:
text_vec_layer = tf.keras.layers.TextVectorization(split="character",
                                                   standardize="lower")
text_vec_layer.adapt([shakespeare_text])
encoded = text_vec_layer([shakespeare_text])[0]

- 시퀀스 투 시퀀스 RNN 훈련을 위해 `윈도의 데이터셋`으로 바뀜

In [6]:
encoded -= 2
# 고유 문자 수 = 39
n_tokens = text_vec_layer.vocabulary_size() - 2
# 총 문자 수 = 1,115,394
dataset_size = len(encoded)

print("n_tokens : ", n_tokens)
print("dataset_size : ", dataset_size)

n_tokens :  39
dataset_size :  1115394


- 예시 :  문자 ID로 구성된 긴 시퀀스를 입력과 타깃 윈도 쌍의 데이터셋으로 변환하는 유틸리티 함수 생성

In [7]:
def to_dataset(sequence, length, shuffle=False, seed=None, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices(sequence)
    ds = ds.window(length + 1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda window_ds: window_ds.batch(length + 1))
    if shuffle:
        ds = ds.shuffle(100_000, seed=seed)
    ds = ds.batch(batch_size)
    return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

- to_dataset()을 사용

In [8]:
list(to_dataset(text_vec_layer(["To be"])[0], length=4))

[(<tf.Tensor: shape=(1, 4), dtype=int64, numpy=array([[ 4,  5,  2, 23]])>,
  <tf.Tensor: shape=(1, 4), dtype=int64, numpy=array([[ 5,  2, 23,  3]])>)]

### 훈련 세트 분리

In [9]:
length = 100
tf.random.set_seed(42)
train_set = to_dataset(encoded[:1_000_000], length=length, shuffle=True,
                       seed=42)
valid_set = to_dataset(encoded[1_000_000:1_060_000], length=length)
test_set = to_dataset(encoded[1_060_000:], length=length)

## Char-RNN 모델 만들기 및 훈련

### GRU층 구축 및 훈련
- 128개의 유닛으로 구성된 GRU층 가진 모델 구축 및 훈련

- `GRU` 클래스의 매개 변수를 이용해 cuDNN 가속 사용
  - activation
  - recurrent_activation
  - recurrent_dropout
  - unroll, use_bias
  - reset_after.

- 첫번째 층 : `Embedding`층으로 문자 ID 인ㄴ코딩

- 출력층에 `Dense`층 사용
  - 39개의 유닛을 사용함 (39개 문자)

- `sparse_categorical_crossentropy` 손실과 `Nadam` 옵티마이저 사용

In [11]:
tf.random.set_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16),
    tf.keras.layers.GRU(128, return_sequences=True),
    tf.keras.layers.Dense(n_tokens, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam",
              metrics=["accuracy"])

model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    "my_shakespeare_model.keras", monitor="val_accuracy", save_best_only=True)

history = model.fit(train_set, validation_data=valid_set, epochs=10,
                    callbacks=[model_ckpt])

Epoch 1/10
  31246/Unknown 411s 12ms/step - accuracy: 0.0661 - loss: nan

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


31247/31247 ━━━━━━━━━━━━━━━━━━━━ 428s 13ms/step - accuracy: 0.0661 - loss: nan - val_accuracy: 0.0691 - val_loss: nan
Epoch 2/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 423s 13ms/step - accuracy: 0.0661 - loss: nan - val_accuracy: 0.0691 - val_loss: nan
Epoch 3/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 469s 14ms/step - accuracy: 0.0661 - loss: nan - val_accuracy: 0.0691 - val_loss: nan
Epoch 4/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 498s 14ms/step - accuracy: 0.0661 - loss: nan - val_accuracy: 0.0691 - val_loss: nan
Epoch 5/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 443s 14ms/step - accuracy: 0.0661 - loss: nan - val_accuracy: 0.0691 - val_loss: nan
Epoch 6/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 501s 14ms/step - accuracy: 0.0661 - loss: nan - val_accuracy: 0.0691 - val_loss: nan
Epoch 7/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 443s 14ms/step - accuracy: 0.0661 - loss: nan - val_accuracy: 0.0691 - val_loss: nan
Epoch 8/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 515s 14ms/step - accuracy: 0.0661 - loss: nan - val_accuracy: 0.

In [20]:
shakespeare_model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Lambda(lambda X: X - 2),
    model
])

- 모델을 이용한 다음 문자 예측

In [24]:
y_proba = shakespeare_model.predict(tf.constant(["To be or not to b"]))[0, -1]
# 가장 가능성이 높은 문자 ID 선택
y_pred = tf.argmax(y_proba)
text_vec_layer.get_vocabulary()[y_pred + 2]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step


np.str_(' ')

## 가짜 셰익스피어 텍스트 생성

### 그리디 코딩
- 초기 텍스트 주입
- 모델이 가장 가능성 있는 다음 글자를 예측
- 예측한 글자를 텍스트 끝에 추가
- 텍스트 모델에 전달하여 다음 글자를 예측

### 온도
- 생성되는 텍스트의 다양성을 제어하는 매개변수
- `높음` :  무작위성 상승 - 정확성 감소
- `낮음` : 무작위성 감소 - 장확성 상승

In [23]:
# 확률 = 50%, 40%, 10%
log_probas = tf.math.log([[0.5, 0.4, 0.1]])
tf.random.set_seed(42)

# 샘플 8개를 추출
tf.random.categorical(log_probas, num_samples=8)

<tf.Tensor: shape=(1, 8), dtype=int64, numpy=array([[0, 0, 1, 1, 1, 0, 0, 0]])>

- 온도를 선택할 수 있는 함수 만들기

In [25]:
def next_char(text, temperature=1):
    y_proba = shakespeare_model.predict([text])[0, -1:]
    rescaled_logits = tf.math.log(y_proba) / temperature
    char_id = tf.random.categorical(rescaled_logits, num_samples=1)[0, 0]
    return text_vec_layer.get_vocabulary()[char_id + 2]

- `next_char()`를 반복적으로 호출하여 다음 글자르 얻는 함수

In [26]:
def extend_text(text, n_chars=50, temperature=1):
    for _ in range(n_chars):
        text += next_char(text, temperature)
    return text

- 온도 테스트
  - 온도 0.01

In [30]:
tf.random.set_seed(42)

print(extend_text(tf.constant(["To be or not to be"]), temperature=0.01))

/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: keras_tensor_4
Received: inputs=('Tensor(shape=(1,))',)
  warnings.warn(msg)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 594ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━

  - 온도 1

In [33]:
print(extend_text(tf.constant(["To be or not to be"]), temperature=1))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━

- 온도 100

In [34]:
print(extend_text(tf.constant(["To be or not to be"]), temperature=100))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━

### 뉴클리어스 샘플링
- 텍스트를 생성하기 위해 상위 `k`개의 문자에서 샘플링
- 확률이 특정 임계값을 초과하는 가장 작은 상위 문자 집합에서만 샘플링
- 단점 : length가 100 이상의 긴 패턴 학습이 어려움
  - 대안 : 상태가 있는 RNN

## 상태가 있는 RNN

### 상태가 없는 RNN
- 훈련 반복마다 모델의 은닉상태를 0으로 초기화
- 상태 업데이트 후 마지막 타임스텝이 필요 없기 때문

### 상태가 있는 RNN
- 훈련 배치(batch)의 마지막 타임 스텝에서 얻은 상태를 다음 훈련 배치의 초기 상태로 사용
- 모델이 장기 패턴을 학습 가능

### RNN을 위한 데이터셋 준비

In [44]:
def to_dataset_for_stateful_rnn(sequence, length):
    ds = tf.data.Dataset.from_tensor_slices(sequence)
    ds = ds.window(length + 1, shift=length, drop_remainder=True)
    ds = ds.flat_map(lambda window: window.batch(length + 1)).batch(1)
    return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

stateful_train_set = to_dataset_for_stateful_rnn(encoded[:1_000_000], length)
stateful_valid_set = to_dataset_for_stateful_rnn(encoded[1_000_000:1_060_000],
                                                 length)
stateful_test_set = to_dataset_for_stateful_rnn(encoded[1_060_000:], length)

In [45]:
list(to_dataset_for_stateful_rnn(tf.range(10), 3))

[(<tf.Tensor: shape=(1, 3), dtype=int32, numpy=array([[0, 1, 2]], dtype=int32)>,
  <tf.Tensor: shape=(1, 3), dtype=int32, numpy=array([[1, 2, 3]], dtype=int32)>),
 (<tf.Tensor: shape=(1, 3), dtype=int32, numpy=array([[3, 4, 5]], dtype=int32)>,
  <tf.Tensor: shape=(1, 3), dtype=int32, numpy=array([[4, 5, 6]], dtype=int32)>),
 (<tf.Tensor: shape=(1, 3), dtype=int32, numpy=array([[6, 7, 8]], dtype=int32)>,
  <tf.Tensor: shape=(1, 3), dtype=int32, numpy=array([[7, 8, 9]], dtype=int32)>)]

### 상태가 잇는 RNN 생성
- `stateful=True`  : 연속적인 배치 준비
- 각 배치 크기 확인 필요 - `input_shape` 매개변수 지정

In [57]:
tf.random.set_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(batch_input_shape=(1, None)),
    tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16),
    tf.keras.layers.GRU(128, return_sequences=True, stateful=True),
    tf.keras.layers.Dense(n_tokens, activation="softmax")
])

- 에포크 끝마다 텍스트를 다시 시작하기 전에 상태를 재설정
  - 사용자 정의 콜백 함수 사용

In [14]:
class ResetStatesCallback(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs):
        self.model.reset_states()

- 체크포인트 저장

In [15]:
model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    "my_stateful_shakespeare_model.keras",
    monitor="val_accuracy",
    save_best_only=True)

- 모델 컴파일 및 콜백 함수 사용

In [58]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
history = model.fit(stateful_train_set, validation_data=stateful_valid_set,
                    epochs=10, callbacks=[ResetStatesCallback(), model_ckpt])

AttributeError: 'Sequential' object has no attribute 'reset_states'

# 2. 감성 분석

## 개요
- 훈련 데이터셋
  - IMDb 영화 리뷰 데이터셋 활용
  - 훈련 세트 / 테스트 세트 25,000개로 구성
  - 레이블링 : 긍정 (1) - 부정(0)

## 데이터셋 로드
- 훈련세트 90% - 검증 10%

In [49]:
import tensorflow_datasets as tfds

raw_train_set, raw_valid_set, raw_test_set = tfds.load(
    name="imdb_reviews",
    split=["train[:90%]", "train[90%:]", "test"],
    as_supervised=True
)
tf.random.set_seed(42)
train_set = raw_train_set.shuffle(5000, seed=42).batch(32).prefetch(1)
valid_set = raw_valid_set.batch(32).prefetch(1)
test_set = raw_test_set.batch(32).prefetch(1)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.Q9W5EW_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.Q9W5EW_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.Q9W5EW_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


- 데이터 확인

In [50]:
for review, label in raw_train_set.take(4):
    print(review.numpy().decode("utf-8")[:200], "...")
    print("레이블:", label.numpy())

This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting  ...
레이블: 0
I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However  ...
레이블: 0
Mann photographs the Alberta Rocky Mountains in a superb fashion, and Jimmy Stewart and Walter Brennan give enjoyable performances as they always seem to do. <br /><br />But come on Hollywood - a Moun ...
레이블: 0
This is the kind of film for a snowy Sunday afternoon when the rest of the world can go ahead with its own business as you descend into a big arm-chair and mellow for a couple of hours. Wonderful perf ...
레이블: 1


## 텍스트 전처리 및 인코딩

### 텍스트 전처리
#### 단어 토큰화
- `단어`를 기준으로 전처리
- `tf.keras.layers.TextVectorization`층 사용
  - 단어 경계를 식별하기 위해 `공백` 사용
  - 따라서 일부 언어 사용 불가
#### 부분 단어 토큰화
- 부분 단어 수준에서 텍스트 토큰화 혹은 복원
- 희귀 단어를 접해도 합리적으로 추측 가능

#### 바이트 페어 인코딩
- 개별 문자로 분할
- 원하는 크기에 도달할 때까지 가장 빈번하게 등장하는 인접 쌍을 반복적으로 병합

#### 부분단어규제
- 훈련중에 토큰화에 약간의 무작위성을 도입

- IMDb 데이터 모델 학습
  - 어휘사전 1000개로 제한
  - 가장 빈번한 998개 단어
  - 패딩 토큰 0
  - 잘 알려지지 않은 단어 1

In [51]:
vocab_size = 1000
text_vec_layer = tf.keras.layers.TextVectorization(max_tokens=vocab_size)
text_vec_layer.adapt(train_set.map(lambda reviews, labels: reviews))

### 모델 학습 및 훈련
- 첫번째 층 : `TextVectorization` 층
- 두번 째 층 : `Embedding`층 - 단어 ID를 임베딩 변환
- 세번 째 층 : `GRU`층 128개 - 단기 기억을 위함
- 네번 째 층 : `Dense`층 - 하나의 뉴런과 시그모이드 활성화 함수

In [52]:
embed_size = 128
tf.random.set_seed(42)
model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Embedding(vocab_size, embed_size),
    tf.keras.layers.GRU(128),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model.compile(loss="binary_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
history = model.fit(train_set, validation_data=valid_set, epochs=2)

Epoch 1/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 26s 34ms/step - accuracy: 0.4937 - loss: 0.6939 - val_accuracy: 0.5020 - val_loss: 0.6929
Epoch 2/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 23s 33ms/step - accuracy: 0.5051 - loss: 0.6930 - val_accuracy: 0.5024 - val_loss: 0.6928


- 한계
  - 일반적으로 모델이 전혀 학습하지 못함
  - 리뷰의 길이가 서로 다르기 때문에 짧은 시퀀스를 패딩 토큰으로 패딩하여 배치해 가장 긴 시퀀스만큼 길제함
  - 많은 패딩 토큰으로 끝남
  - GRU층을 사용해도 단기기억이 좋지 안흥ㅁ

- 대안
  - 모델에 동일한 길이의 문자으로 구성된 배치를 주입
  - RNN이 패딩 토큰을 무시 -> `마스킹`

## 마스킹
> 패팅 토큰(ID=0)이 모델 학습에 영향을 주지 않도록 무시

### 필요성
TextVectorization은 가변 길이 시퀀스를 위해 패딩 토큰을 추가하는데, 이 패딩은 GRU 층의 평균 계산 등을 왜곡할 수 있음

### 케라스로 구현
- `Embedding`층을 만들 때, `mask_zero=True`를 매개변수로 추가
  - 모든 층에서 패딩 토큰을 무시
- `supports_maksing`속성
  - 마스크가 자동으로 다음층으로 전파
- `return_sequences=False` : 마지막 순환층까지 자동으로 전파

In [53]:
embed_size = 128
tf.random.set_seed(42)
model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True),
    tf.keras.layers.GRU(128),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model.compile(loss="binary_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
history = model.fit(train_set, validation_data=valid_set, epochs=5)

Epoch 1/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 28s 35ms/step - accuracy: 0.6298 - loss: 0.6274 - val_accuracy: 0.7460 - val_loss: 0.5287
Epoch 2/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 25s 35ms/step - accuracy: 0.8338 - loss: 0.3826 - val_accuracy: 0.8712 - val_loss: 0.3074
Epoch 3/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 24s 34ms/step - accuracy: 0.8769 - loss: 0.2948 - val_accuracy: 0.8788 - val_loss: 0.3021
Epoch 4/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 24s 34ms/step - accuracy: 0.8901 - loss: 0.2690 - val_accuracy: 0.8664 - val_loss: 0.3341
Epoch 5/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 43s 37ms/step - accuracy: 0.8979 - loss: 0.2504 - val_accuracy: 0.8664 - val_loss: 0.3191


- 함수형 API 혹은 서브클래싱 API로 마스크를 명시적으로 계산

In [60]:
tf.random.set_seed(42)
inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)
token_ids = text_vec_layer(inputs)
mask = tf.keras.ops.not_equal(token_ids, 0)
Z = tf.keras.layers.Embedding(vocab_size, embed_size)(token_ids)
Z = tf.keras.layers.GRU(128, dropout=0.2)(Z, mask=mask)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(Z)
model = tf.keras.Model(inputs=[inputs], outputs=[outputs])

- 래그드 텐서 주입

In [61]:
text_vec_layer_ragged = tf.keras.layers.TextVectorization(
    max_tokens=vocab_size, ragged=True)
text_vec_layer_ragged.adapt(train_set.map(lambda reviews, labels: reviews))
text_vec_layer_ragged(["Great movie!", "This is DiCaprio's best role."])

<tf.RaggedTensor [[86, 18], [11, 7, 1, 116, 217]]>

- 래그드 텐서 표현을 패딩 토큰을 사용하여 일반 텐서 표현과 비교

In [62]:
text_vec_layer(["Great movie!", "This is DiCaprio's best role."])

<tf.Tensor: shape=(2, 5), dtype=int64, numpy=
array([[ 86,  18,   0,   0,   0],
       [ 11,   7,   1, 116, 217]])>

## 사전 훈련 임베딩과 언어 모델 재사용

### 사전 훈련 임베딩
- 대규모 텍스트 데이터셋으로 미리 학습된 단어 임베딩 재사용
- 소규모 데이터셋에서 모델 성능을 크게 향상 가능
- 예시 : Word2Vec, GloVe, FastText 등

### 문맥 임베딩
- 단어가 나타나는 문맥에 따라 임베딩 벡터가 달라지는 것

### ULMFit
- 사전 훈련된 언어 모델을 전이 학습하여 다양한 NLP 작업에 적용
- 소규모 데이터셋만으로도 모델 성능달 성 가능
- `Universal Sentence Encoder`
  - 문장 전체를 하나의 고정된 크기 벡터로 임베딩

In [65]:
import os
import tensorflow_hub as hub

os.environ["TFHUB_CACHE_DIR"] = "my_tfhub_cache"
tf.random.set_seed(42)
model = tf.keras.Sequential([
    hub.KerasLayer("https://tfhub.dev/google/universal-sentence-encoder/4",
                   trainable=False, dtype=tf.string, input_shape=[]),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model.compile(loss="binary_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
model.fit(train_set, validation_data=valid_set, epochs=10)

ValueError: Only instances of `keras.Layer` can be added to a Sequential model. Received: <tensorflow_hub.keras_layer.KerasLayer object at 0x79038dc71e90> (of type <class 'tensorflow_hub.keras_layer.KerasLayer'>)